In [0]:
import json
from pathlib import Path

import pandas as pd

In [0]:
dbutils.widgets.text(
    "raw_root",
    "/Volumes/workspace/raw/usgs_earthquakes",
    "Raw Root"
)

raw_root = Path(
    dbutils.widgets.get("raw_root")
)

In [0]:
%sql

DROP DATABASE IF EXISTS workspace.bronze CASCADE;

In [0]:
%sql

CREATE DATABASE IF NOT EXISTS workspace.bronze
COMMENT 'Capa Bronze: datos crudos estructurados USGS';

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.usgs_earthquakes (
    earthquake_id STRING,
    magnitude DOUBLE,
    place STRING,
    event_time BIGINT,
    updated_time BIGINT,
    felt BIGINT,
    cdi DOUBLE,
    mmi DOUBLE,
    alert STRING,
    status STRING,
    tsunami INT,
    significance BIGINT,
    network STRING,
    event_code STRING,
    ids STRING,
    sources STRING,
    product_types STRING,
    station_count BIGINT,
    distance_min DOUBLE,
    rms DOUBLE,
    azimuthal_gap DOUBLE,
    magnitude_type STRING,
    event_type STRING,
    longitude DOUBLE,
    latitude DOUBLE,
    depth DOUBLE
)
""")

In [0]:
candidate_files = sorted(
    raw_root.glob(
        "**/usgs_earthquakes.geojson"
    )
)

records = []

for json_file in candidate_files:

    payload = json.loads(
        json_file.read_text(
            encoding="utf-8"
        )
    )

    features = payload.get(
        "features",
        []
    )

    for feature in features:

        properties = feature.get(
            "properties",
            {}
        )

        geometry = feature.get(
            "geometry",
            {}
        )

        coordinates = geometry.get(
            "coordinates",
            [None, None, None]
        )

        longitude = (
            coordinates[0]
            if len(coordinates) > 0
            else None
        )

        latitude = (
            coordinates[1]
            if len(coordinates) > 1
            else None
        )

        depth = (
            coordinates[2]
            if len(coordinates) > 2
            else None
        )

        records.append({

            "earthquake_id":
                feature.get("id"),

            "magnitude":
                properties.get("mag"),

            "place":
                properties.get("place"),

            "event_time":
                properties.get("time"),

            "updated_time":
                properties.get("updated"),

            "felt":
                properties.get("felt"),

            "cdi":
                properties.get("cdi"),

            "mmi":
                properties.get("mmi"),

            "alert":
                properties.get("alert"),

            "status":
                properties.get("status"),

            "tsunami":
                properties.get("tsunami"),

            "significance":
                properties.get("sig"),

            "network":
                properties.get("net"),

            "event_code":
                properties.get("code"),

            "ids":
                properties.get("ids"),

            "sources":
                properties.get("sources"),

            "product_types":
                properties.get("types"),

            "station_count":
                properties.get("nst"),

            "distance_min":
                properties.get("dmin"),

            "rms":
                properties.get("rms"),

            "azimuthal_gap":
                properties.get("gap"),

            "magnitude_type":
                properties.get("magType"),

            "event_type":
                properties.get("type"),

            "longitude":
                longitude,

            "latitude":
                latitude,

            "depth":
                depth
        })

In [0]:
print(
    f"Archivos encontrados: {len(candidate_files)}"
)

print(
    f"Eventos encontrados: {len(records)}"
)

In [0]:
df = pd.DataFrame(records)

df_spark = spark.createDataFrame(
    df
)

display(df_spark)

In [0]:
from pyspark.sql.functions import col

df_spark_fixed = df_spark \
    .withColumn("felt", col("felt").cast("bigint")) \
    .withColumn("station_count", col("station_count").cast("bigint")) \
    .withColumn("tsunami", col("tsunami").cast("int"))

df_spark_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "mergeSchema",
        "true"
    ) \
    .saveAsTable(
        "workspace.bronze.usgs_earthquakes"
    )